# Monthly workflow

The README's terminal commands are the main way to run the month
(`trex parse …`, `trex reconcile …`, `trex summarize --push`); this notebook
does the same steps, then has a section for exploring the data.

Drop the new statement PDFs into `data/statements/`, along with any new IHP
claims export, then run top to bottom. Clear all outputs before committing:
the pre-commit hook rejects a notebook that has them, since they would hold real
transactions.

In [ ]:
from trex import (
    build_year_summary,
    check_year_summary_total,
    parse_statement,
    reconcile_in_place,
    summarize,
)
from trex.log import configure_logging
from trex.parse import check_year_balances
from trex.summary import read_year_expenses, update_statement_totals

configure_logging()  # print trex's messages under each cell; verbose=True for debug
YEAR = 2026

## 1. Parse the new statements

Short names of this month's PDFs. Each is extracted, categorized, balance-checked
and written to `data/parsed/<name>.csv`.

In [ ]:
NEW = ["Chase05", "PLG05", "PLY0405", "UOB05"]  # this month's statements

for name in NEW:
    parse_statement(name, year=YEAR)

unbalanced = {n: m for n, m in check_year_balances(year=YEAR).items() if n in NEW}
assert not unbalanced, f"don't balance, fix before going on: {sorted(unbalanced)}"

## 2. Fix miscategorized rows

In `data/parsed/<name>.csv`, edit the categories cell of any wrong row, including
those in the `uncategorized` and `multi-category` sections. Then fold the fixes
into `cat.csv` and re-parse (`trex reconcile`):

In [ ]:
# reconcile_in_place(["UOB05"])

## 3. Summarize the year

Same as `trex summarize`: writes `data/summary/`, records each statement's
total, then runs every check the CLI fails on.

In [ ]:
summarize(year=YEAR)
changes = update_statement_totals(year=YEAR)  # accept=True to take changes
unbalanced = check_year_balances(year=YEAR)
_, _, difference = check_year_summary_total(year=YEAR)
summary = build_year_summary(YEAR, read_year_expenses(YEAR))

failed = {
    check: value
    for check, value in {
        "changed statement totals": len(changes),
        "unbalanced statements": sorted(unbalanced),
        "summary total off by": round(difference, 2) if abs(difference) >= 0.01 else 0,
        "uncategorized rows": summary.uncategorized.rows,
        "multi-category rows": summary.multi_category.rows,
    }.items()
    if value
}
failed or "all checks pass"

## 4. Commit

If every check passes, commit and push the month from a terminal:

```bash
trex summarize --push
```

# Explore

The year's rows as DataFrames. `cost` is as printed (USD for the Chase card),
`sgd` is converted. Use `df["sgd"].to_numpy()` for a plain array.

In [ ]:
import pandas as pd

from trex.config import get_parsed_dir, get_summary_dir
from trex.constants import CATEGORIES, HEALTHCARE_CATEGORY_ID
from trex.healthcare import healthcare_listing_name, read_healthcare_csv
from trex.serialize import read_parsed_expenses
from trex.summary import read_year_credits, read_year_expenses, to_sgd

expenses = pd.DataFrame(read_year_expenses(YEAR))
expenses["sgd"] = expenses.apply(lambda row: to_sgd(row.cost, row.card), axis=1)
expenses["category"] = expenses.category_ids.map(
    lambda ids: ", ".join(CATEGORIES[i] for i in ids) or "-"
)
credits = pd.DataFrame(read_year_credits(YEAR))
listing = get_summary_dir() / healthcare_listing_name(YEAR)
healthcare = pd.DataFrame(read_healthcare_csv(listing) if listing.exists() else [])
expenses.head()

Starting points. Rows that are uncategorized or multi-category count toward no
category total; this lists them, with the parsed file to edit.

In [ ]:
expenses[expenses.category_ids.map(len) != 1][["statement", "date", "remark", "sgd", "category"]]

In [ ]:
expenses.pivot_table(
    index="category", columns="month", values="sgd", aggfunc="sum", fill_value=0, margins=True
).round(2)

In [ ]:
expenses.groupby("card").sgd.sum().round(2).sort_values(ascending=False)

In [ ]:
expenses.groupby("remark").sgd.agg(["count", "sum"]).sort_values("sum", ascending=False).head(20)

In [ ]:
expenses[expenses.remark.str.contains("GRAB", case=False)]

In [ ]:
# one statement, as parsed
pd.DataFrame(read_parsed_expenses(get_parsed_dir() / "UOB05.csv"))